# MathForge 7B — full 2,956-row QLoRA

Fresh one-epoch 4-bit LoRA training over all 2,956 unique, integrity-audited problems. This is a separate broad-data experiment and will not resume or overwrite the creative-205 adapter. Use an **A100** when possible; an **L4** will be substantially slower.

## 1. Verify the GPU runtime

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > GPU.'
GPU_NAME = torch.cuda.get_device_name(0)
assert any(name in GPU_NAME for name in ('A100', 'L4')), f'Use an A100 or L4, not {GPU_NAME}.'
assert torch.cuda.is_bf16_supported(), f'{GPU_NAME} must support BF16.'
print('GPU:', GPU_NAME)

## 2. Install the pinned stack

In [ ]:
!pip install -q "transformers==5.13.0" "trl==1.8.0" "peft==0.19.1" "datasets==5.0.0" "accelerate==1.14.0" "bitsandbytes==0.49.2"

## 3. Mount Drive and upload `train.jsonl`
Choose `/Users/arthurxu/Desktop/SLM/data/train.jsonl` when prompted.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly train.jsonl.'
DATA = next(iter(uploaded))
print('uploaded:', DATA)

## 4. Validate and freeze the experiment

In [ ]:
import hashlib, json, os
from pathlib import Path

RUN_ID = 'full2956-20260712-v1'
EXPECTED_SHA256 = '79a4b898d433d9e8517dfb85f3049fd67139a1156550c4caa071ec8ade8e2759'
EXPECTED_ROWS = 2956
BASE_MODEL = 'Qwen/Qwen2.5-Math-7B-Instruct'
OUT = f'/content/drive/MyDrive/mathforge-qlora-7b-{RUN_ID}'
EPOCHS = 1
MAX_SEQ_LEN = 1536
LR = 2e-4
LORA_R = 32
GRAD_ACCUM = 8
SAVE_STEPS = 50
SEED = 42

payload = Path(DATA).read_bytes()
actual_hash = hashlib.sha256(payload).hexdigest()
rows = [json.loads(line) for line in payload.decode('utf-8').splitlines() if line.strip()]
ids = [str((row.get('meta') or {}).get('id') or '') for row in rows]
assert actual_hash == EXPECTED_SHA256, (actual_hash, EXPECTED_SHA256)
assert len(rows) == EXPECTED_ROWS and len(set(ids)) == EXPECTED_ROWS and all(ids)
assert all([m.get('role') for m in row.get('messages', [])] == ['user', 'assistant'] for row in rows)

manifest = {
    'run_id': RUN_ID, 'dataset_sha256': actual_hash, 'rows': len(rows),
    'base_model': BASE_MODEL, 'epochs': EPOCHS, 'max_seq_len': MAX_SEQ_LEN,
    'learning_rate': LR, 'lora_rank': LORA_R, 'grad_accum': GRAD_ACCUM,
    'save_steps': SAVE_STEPS, 'seed': SEED,
    'packages': {
        'transformers': '5.13.0', 'trl': '1.8.0', 'peft': '0.19.1',
        'datasets': '5.0.0', 'accelerate': '1.14.0', 'bitsandbytes': '0.49.2',
    },
}
os.makedirs(OUT, exist_ok=True)
manifest_path = Path(OUT) / 'run_manifest.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest, 'Existing checkpoints use another configuration.'
else:
    manifest_path.write_text(json.dumps(manifest, indent=2) + '\n')
print(f'PASS: {len(rows)} unique rows; sha256={actual_hash}; OUT={OUT}')

## 5. Train one epoch
Expect about 359 optimizer steps. Checkpoints are written every 50 steps; final weights and a final evaluation are always saved.

In [ ]:
import os, torch
from collections import Counter
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.trainer_utils import get_last_checkpoint
from trl import SFTConfig, SFTTrainer

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map='auto', torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
model.gradient_checkpointing_enable()
lora = LoraConfig(
    r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.05, bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)

ds = load_dataset('json', data_files=DATA, split='train')
ids = [str((row.get('meta') or {}).get('id') or '') for row in ds]
duplicates = [pid for pid, count in Counter(ids).items() if not pid or count > 1]
assert not duplicates and len(ds) == EXPECTED_ROWS
split = ds.train_test_split(test_size=0.03, seed=SEED)
train_ids = {row['meta']['id'] for row in split['train']}
eval_ids = {row['meta']['id'] for row in split['test']}
assert not (train_ids & eval_ids)

def format_row(row):
    return {'text': tok.apply_chat_template(row['messages'], tokenize=False)}

train_ds = split['train'].map(format_row, remove_columns=split['train'].column_names)
eval_ds = split['test'].map(format_row, remove_columns=split['test'].column_names)
print(f'train={len(train_ds)} eval={len(eval_ds)}')
cfg = SFTConfig(
    output_dir=OUT, num_train_epochs=EPOCHS,
    per_device_train_batch_size=1, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, warmup_ratio=0.05, lr_scheduler_type='cosine',
    logging_steps=10, eval_strategy='steps', eval_steps=SAVE_STEPS,
    save_strategy='steps', save_steps=SAVE_STEPS, save_total_limit=3,
    load_best_model_at_end=False, bf16=True, max_length=MAX_SEQ_LEN,
    packing=False, dataset_text_field='text', gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    seed=SEED, report_to='none',
)
trainer = SFTTrainer(
    model=model, args=cfg, train_dataset=train_ds, eval_dataset=eval_ds, peft_config=lora,
)
resume = get_last_checkpoint(OUT) if os.path.isdir(OUT) else None
if resume:
    print('resuming:', resume)
result = trainer.train(resume_from_checkpoint=resume)
final_eval = trainer.evaluate()
trainer.save_model(OUT)
trainer.save_state()
tok.save_pretrained(OUT)
trainer.save_metrics('train', result.metrics)
trainer.save_metrics('eval', final_eval)
print('steps:', trainer.state.global_step, '| saved adapter:', OUT)

## 6. Smoke test

In [ ]:
eval_model = trainer.model
eval_model.config.use_cache = True
eval_model.eval()
prompt = (
    'Compose one original, elegant AIME-level number theory problem with a single integer answer in [0, 999]. '
    'Target difficulty (AoPS 1-10): 6.5. Target problem-elegance (0-5): 4.5. '
    'Give the statement, then a clean solution ending in the integer answer.'
)
inputs = tok.apply_chat_template(
    [{'role': 'user', 'content': prompt}], add_generation_prompt=True,
    return_tensors='pt', return_dict=True,
).to(eval_model.device)
torch.manual_seed(123); torch.cuda.manual_seed_all(123)
with torch.no_grad():
    output = eval_model.generate(**inputs, max_new_tokens=1200, do_sample=True, temperature=0.9, top_p=0.95)
print(tok.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))